# Ch.7 — Contrastive Learning (SimCLR, MoCo)

**The breakthrough:** Learn visual representations from unlabeled data by contrasting augmented views.

**What you'll build:** SimCLR contrastive pretraining on unlabeled images, fine-tune on labeled detection task.

**Dataset:** Synthetic retail shelf dataset (SmallVal AI) — 50k unlabeled + 1k labeled images.

## 1. Setup and Imports

In [ ]:
# TODO: Implement this cell


## 2. Contrastive Augmentation Pipeline

SimCLR's key insight: **strong augmentations** are critical. Without them, the model collapses.

We implement the full augmentation stack from Chen et al. (2020):
- Random crop (0.08–1.0 scale)
- Color jitter (brightness, contrast, saturation, hue)
- Random grayscale
- Gaussian blur
- Random horizontal flip

In [ ]:
# TODO: Implement this cell


## 3. SimCLR Model Architecture

**Components:**
1. **Encoder** $f(\cdot)$: ResNet-50 (remove final FC layer) → outputs 2048-d features
2. **Projection Head** $g(\cdot)$: MLP (2048 → 2048 → 128) → outputs 128-d embeddings

**Key insight:** Projection head is discarded after pretraining! Only encoder features are used for downstream tasks.

In [ ]:
# TODO: Implement this cell


## 4. NT-Xent Loss (Normalized Temperature-Scaled Cross Entropy)

$$
\mathcal{L}_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k=1}^{2N} \mathbb{1}_{k \neq i} \exp(\text{sim}(z_i, z_k) / \tau)}
$$

**Intuition:** Treat contrastive learning as a classification problem where:
- **Positive class**: The other augmented view of the same image (1 sample)
- **Negative classes**: All other images in the batch (2N-2 samples)

The loss pulls positive pairs together, pushes negative pairs apart.

In [ ]:
# TODO: Implement this cell


## 5. SimCLR Pretraining Loop

**Hyperparameters (from Chen et al., 2020):**
- Batch size: 256 (512 views)
- Temperature: 0.07
- Optimizer: LARS (we use SGD with momentum as approximation)
- Learning rate: 0.3 × batch_size / 256 (linear scaling rule)
- Epochs: 800 (we train for 10 for demonstration)

**Training time:** ~3 days on 4× V100 GPUs for full 800 epochs on 50k images.

In [ ]:
# TODO: Implement this cell


## 6. Visualize Training Progress

In [ ]:
# TODO: Implement this cell


## 7. Fine-Tuning on Labeled Data

After pretraining, we:
1. **Discard projection head** (only needed for contrastive loss)
2. **Keep encoder** (ResNet-50 with learned features)
3. **Attach detection head** (e.g., YOLO, Faster R-CNN)
4. **Fine-tune on 1k labeled images**

**Result:** Achieve 84% mAP (vs 72% from scratch) with same 1k labels!

In [ ]:
# TODO: Implement this cell


## 8. Data Efficiency Comparison

**Key result:** Contrastive pretraining enables data-efficient fine-tuning.

| Training Strategy | Labeled Images | mAP@0.5 | Labeling Cost |
|-------------------|----------------|---------|---------------|
| From scratch | 1,000 | 72% | $5k |
| From scratch | 10,000 | 85% ✅ | $50k |
| **SimCLR pretrained** | **1,000** | **84%** | **$5k** |
| SimCLR + more labels | 2,000 | 87% | $10k |

**Insight:** Contrastive pretraining gives +12% mAP gain at same label count!

In [ ]:
# TODO: Implement this cell


## 9. Exercises

**Exercise 1:** Implement momentum encoder for MoCo  
Modify the training loop to use a momentum encoder and queue of negatives.

**Exercise 2:** Temperature sweep  
Train with τ ∈ {0.05, 0.07, 0.1, 0.3, 0.5} and compare downstream performance.

**Exercise 3:** Augmentation ablation  
Remove color jitter or blur from the augmentation pipeline and observe model collapse.

In [ ]:
# TODO: Implement this cell
